In [ ]:
import sys
from difflib import SequenceMatcher
from pathlib import Path

from datasets import load_from_disk

ds = load_from_disk("../data/processed/hf_dataset")

print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'sql', 'famille', 'difficulte'],
        num_rows: 492
    })
    validation: Dataset({
        features: ['question', 'sql', 'famille', 'difficulte'],
        num_rows: 100
    })
    test: Dataset({
        features: ['question', 'sql', 'famille', 'difficulte'],
        num_rows: 100
    })
})
{'question': "Affiche le top 3 des types de forfaits selon leur part dans le chiffre d'affaires total payé", 'sql': "SELECT a.type_forfait, 100.0 * SUM(f.montant_paye) / (\n  SELECT SUM(montant_paye)\n  FROM factures\n  WHERE statut_paiement = 'Payée'\n)\nFROM abonnements a\nJOIN factures f ON a.abonnement_id = f.abonnement_id\nWHERE f.statut_paiement = 'Payée'\nGROUP BY a.type_forfait\nORDER BY SUM(f.montant_paye) DESC\nLIMIT 3;", 'famille': 'Ratios et pourcentages', 'difficulte': 'difficile'}


In [49]:
print(Path(sys.path[0]).resolve())
print(ROOT)

/Users/Akhenaton/Documents/Projets_Portfolio/text-to-sql-agent
/Users/Akhenaton


In [52]:
ROOT = Path(sys.path[0]).resolve()

ds = load_from_disk(ROOT / "data" / "processed" / "hf_dataset")
ref = [ex["question"].lower() for s in ("train", "validation") for ex in ds[s]]

for ex in ds["test"]:
    q = ex["question"].lower()
    best = max(ref, key=lambda r: SequenceMatcher(None, q, r).ratio())
    score = SequenceMatcher(None, q, best).ratio()
    if score > 0.85:
        print(f"{score:.2f} | TEST : {ex['question']}\n       | TRAIN: {best}\n")

0.88 | TEST : Quels clients ont une adresse email contenant gmail ?
       | TRAIN: quels clients ont une adresse email contenant le mot lopes ?

0.85 | TEST : Quelles factures ont été émises en juin 2026 ?
       | TRAIN: quelles factures ont été créées en 2025 ?

0.86 | TEST : Quels clients se sont inscrits un lundi ?
       | TRAIN: quels clients se sont inscrits en 2026 ?

